In [2]:
# %% Shared setup (standard library only - NO pandas / NumPy)
import csv, json, shutil
from pathlib import Path
from collections import Counter, defaultdict

LAST_NAME = "Tapiculin"
STUDENT_ID = "TUPM-26-1932"
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"          # put the Kaggle CSV files here (git-ignored)
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"
for d in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    d.mkdir(exist_ok=True)
print("Seed:", SEED_DIGIT, "| ID Sum:", ID_SUM, "| Name Length:", NAME_LENGTH)

# %% EXERCISE 3 - Cybersecurity Incident Report (Global_Cybersecurity_Threats_2015-2024.csv)
cyber_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
TARGET_YEAR = 2015 + SEED_DIGIT
LOSS = "Financial Loss (in Million $)"

total_rows = 0
incidents, bad_loss = [], 0
with open(cyber_path, "r", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        total_rows += 1
        try:
            if int(row["Year"]) != TARGET_YEAR:
                continue
            loss = float(row[LOSS])
        except (ValueError, KeyError):             # invalid financial values
            bad_loss += 1
            continue
        incidents.append((row, loss))

attack_counts = Counter(r["Attack Type"] for r, _ in incidents)
total_loss = sum(l for _, l in incidents)
avg_loss = total_loss / len(incidents) if incidents else 0.0
top5 = sorted(incidents, key=lambda x: -x[1])[:5]  # stable: ties keep original dataset order

top5_json = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_top5_incidents.json"
with open(top5_json, "w", encoding="utf-8") as f:
    json.dump([dict(r, **{LOSS: l}) for r, l in top5], f, indent=4)

filtered_csv = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_filtered.csv"
if incidents:
    with open(filtered_csv, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(incidents[0][0].keys())); w.writeheader()
        w.writerows(r for r, _ in incidents)
backup_json = BACKUP_DIR / top5_json.name
shutil.copy2(top5_json, backup_json)

report = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_report.txt"
with open(report, "w", encoding="utf-8") as f:
    f.write(f"CYBERSECURITY INCIDENT REPORT - {TARGET_YEAR}\n")
    f.write(f"Incidents: {len(incidents)}\nTotal loss (USD M): {total_loss:.2f}\nAverage loss (USD M): {avg_loss:.2f}\n")
    f.write(f"Attack types: {dict(attack_counts)}\nFive costliest incidents:\n")
    for r, l in top5:
        f.write(f"  {r['Attack Type']} | {r.get('Country','')} | {l}\n")
    f.write(f"Output files: {filtered_csv.name}, {top5_json.name}, {backup_json.name}\n")

print("Target year:", TARGET_YEAR)
print("Total records in dataset:", total_rows)
print("Incidents in target year:", len(incidents), "(invalid loss values skipped:", bad_loss, ")")
print("Count per attack type:", dict(attack_counts))
print("Total financial loss (USD millions):", round(total_loss, 2))
print("Average financial loss (USD millions):", round(avg_loss, 2))
print("Five costliest incidents:", [(r["Attack Type"], l) for r, l in top5])
print("Filtered CSV file:", filtered_csv)
print("JSON summary file:", top5_json)
print("Backup verification:", backup_json.exists())
print("Report file:", report)

Seed: 2 | ID Sum: 23 | Name Length: 9
Target year: 2017
Total records in dataset: 3000
Incidents in target year: 319 (invalid loss values skipped: 0 )
Count per attack type: {'Man-in-the-Middle': 58, 'DDoS': 58, 'Phishing': 64, 'SQL Injection': 49, 'Malware': 46, 'Ransomware': 44}
Total financial loss (USD millions): 16261.68
Average financial loss (USD millions): 50.98
Five costliest incidents: [('SQL Injection', 99.99), ('Malware', 99.22), ('SQL Injection', 98.75), ('DDoS', 98.73), ('DDoS', 98.61)]
Filtered CSV file: output\Tapiculin_TUPM-26-1932_cyber_filtered.csv
JSON summary file: output\Tapiculin_TUPM-26-1932_top5_incidents.json
Backup verification: True
Report file: output\Tapiculin_TUPM-26-1932_cyber_report.txt
